# AICE Associate 대비 실습 교재
## Chapter 02. 범주형 데이터 전처리: 원-핫 인코딩과 레이블 인코딩

본 교재는 **AICE Associate** 실기 시험의 핵심 평가 요소인 **범주형(Categorical) 데이터 전처리**를 다룹니다.
머신러닝 모델이 이해할 수 있도록 문자로 된 범주형 변수를 수치형으로 변환하는 **One-Hot Encoding**과 **Label Encoding**의 개념 및 실습을 진행합니다.

---

### 📋 학습 목차
1. **Section 01. 범주형 데이터의 이해와 분류** (명목형 vs 순서형)
2. **Section 02. 원-핫 인코딩 (One-Hot Encoding)** (`pd.get_dummies()`, `OneHotEncoder`)
3. **Section 03. 레이블 인코딩 (Label Encoding)** (`LabelEncoder`)
4. **Section 04. AICE 시험 실전 유형 연습** (범주형 변수 처리 및 통합 전처리)

## 00. 실습 환경 설정 및 샘플 데이터 생성
실습에 필요한 Python 패키지를 임포트하고, 범주형 변수가 포함된 고객/제품 샘플 데이터셋(`sample_encoding.csv`)을 생성합니다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
'''
set root directory according your environment
'''
ROOT_DIR = '/content/drive/MyDrive/Colab Notebooks/AS_260824'

In [4]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# 샘플 데이터 생성 (고객 정보 및 구매 등급)
df_raw = pd.DataFrame({
    'CustomerID': ['C001', 'C002', 'C003', 'C004', 'C005', 'C006'],
    'City': ['서울', '부산', '서울', '대구', '부산', '광주'],       # 명목형 (순서 없음)
    'Education': ['학사', '석사', '고졸', '박사', '학사', '석사'], # 명목/순서형
    'Satisfaction': ['보통', '만족', '불만족', '매우만족', '만족', '보통'], # 순서형 (크기 관계 존재)
    'PurchaseAmount': [150, 230, 80, 450, 310, 190]
})
display(df_raw)
# CSV 파일 저장
df_raw.to_csv(os.path.join(ROOT_DIR, 'sample_encoding.csv'), index=False, encoding='utf-8-sig')
print('✅ 실습용 데이터 파일 생성 완료: sample_encoding.csv')

,CustomerID,City,Education,Satisfaction,PurchaseAmount
0,C001,서울,학사,보통,150
1,C002,부산,석사,만족,230
2,C003,서울,고졸,불만족,80
3,C004,대구,박사,매우만족,450
4,C005,부산,학사,만족,310
5,C006,광주,석사,보통,190


✅ 실습용 데이터 파일 생성 완료: sample_encoding.csv


---
## Section 01. 범주형 데이터 전처리 개요

범주형 데이터는 머신러닝 알고리즘에 직접 입력할 수 없으므로 수치화하는 작업이 필수적입니다.

| 인코딩 방식 | 설명 | 주요 사용 상황 | 시험 중요도 |
|---|---|---|---|
| **One-Hot Encoding** | 각 범주를 0과 1의 이진 벡터로 변환 (컬럼 추가) | 순서가 없는 명목형 변수 (예: 도시, 성별) | ★★★★★ |
| **Label Encoding** | 각 범주를 0, 1, 2... 식의 정수로 변환 | 순서가 있는 변수 또는 나무 기반 모델 (Decision Tree 등) | ★★★★☆ |

> **⚠️ 주의점**: 명목형 변수에 Label Encoding을 적용하면 모델이 숫자 간의 크기 관계(예: 2 > 1)로 오해할 수 있으므로 선형 모델에서는 One-Hot Encoding을 권장합니다.

---
## Section 02. 원-핫 인코딩 (One-Hot Encoding)

### 1. `pd.get_dummies()` 활용 (Pandas 방식)
AICE 시험에서 가장 빠르고 간편하게 원-핫 인코딩을 적용하는 방법입니다.

In [5]:
df = pd.read_csv(os.path.join(ROOT_DIR, 'sample_encoding.csv'))
df

,CustomerID,City,Education,Satisfaction,PurchaseAmount
0,C001,서울,학사,보통,150
1,C002,부산,석사,만족,230
2,C003,서울,고졸,불만족,80
3,C004,대구,박사,매우만족,450
4,C005,부산,학사,만족,310
5,C006,광주,석사,보통,190


In [7]:
# City 컬럼 원-핫 인코딩 적용
df_encoded = pd.get_dummies(df, columns=['City'], dtype=int)
print('--- [pd.get_dummies() 적용 결과] ---')
display(df_encoded.head())

--- [pd.get_dummies() 적용 결과] ---


,CustomerID,Education,Satisfaction,PurchaseAmount,City_광주,City_대구,City_부산,City_서울
0,C001,학사,보통,150,0,0,0,1
1,C002,석사,만족,230,0,0,1,0
2,C003,고졸,불만족,80,0,0,0,1
3,C004,박사,매우만족,450,0,1,0,0
4,C005,학사,만족,310,0,0,1,0


### 2. `sklearn.preprocessing.OneHotEncoder` 활용 (Scikit-Learn 방식)
파이프라인 구축이나 학습/검증 데이터셋 분리 시 변환 기준을 유지하기 위해 사용합니다.

In [10]:
ohe = OneHotEncoder(sparse_output=False)
city_encoded = ohe.fit_transform(df[['City']])
display(city_encoded)
print(ohe.get_feature_names_out(['City']))
# 변환된 결과를 DataFrame으로 변환
df_ohe = pd.DataFrame(city_encoded, columns=ohe.get_feature_names_out(['City']))
print('--- [OneHotEncoder 적용 결과] ---')
display(df_ohe)

array([[0., 0., 0., 1.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [1., 0., 0., 0.]])

['City_광주' 'City_대구' 'City_부산' 'City_서울']
--- [OneHotEncoder 적용 결과] ---


,City_광주,City_대구,City_부산,City_서울
0,0.0,0.0,0.0,1.0
1,0.0,0.0,1.0,0.0
2,0.0,0.0,0.0,1.0
3,0.0,1.0,0.0,0.0
4,0.0,0.0,1.0,0.0
5,1.0,0.0,0.0,0.0


---
## Section 03. 레이블 인코딩 (Label Encoding)

`sklearn.preprocessing.LabelEncoder`를 사용하여 각 문자를 정수값으로 1:1 매핑합니다.

In [13]:
#df_label = pd.read_csv('sample_encoding.csv')
df_label = df_raw.copy()
display(df_label)
# LabelEncoder 객체 생성 및 적용
le = LabelEncoder()
df_label['Satisfaction_encoded'] = le.fit_transform(df_label['Satisfaction'])

print('--- [LabelEncoder 적용 결과] ---')
display(df_label[['Satisfaction', 'Satisfaction_encoded']])
print('\n매핑된 클래스 목록:', le.classes_)

,CustomerID,City,Education,Satisfaction,PurchaseAmount
0,C001,서울,학사,보통,150
1,C002,부산,석사,만족,230
2,C003,서울,고졸,불만족,80
3,C004,대구,박사,매우만족,450
4,C005,부산,학사,만족,310
5,C006,광주,석사,보통,190


--- [LabelEncoder 적용 결과] ---


,Satisfaction,Satisfaction_encoded
0,보통,2
1,만족,0
2,불만족,3
3,매우만족,1
4,만족,0
5,보통,2



매핑된 클래스 목록: ['만족' '매우만족' '보통' '불만족']


---
## Section 04. AICE 시험 실전 유형 연습

### 🎯 실전 문제
1. `sample_encoding.csv` 데이터를 로딩하세요.
2. `Education` 컬럼에 대해 **Label Encoding**을 적용하여 `Education_Label` 컬럼을 신규 생성하세요.
3. `City` 컬럼에 대해 **Pandas `get_dummies()`**를 사용하여 원-핫 인코딩을 수행하고, 기존 `City` 컬럼은 변환 결과로 대체하세요.
4. 최종 완성된 데이터프레임의 `head()`를 확인하세요.

In [15]:
# [실전 풀이]
# Step 1. 데이터 로딩
#df_exam = pd.read_csv('sample_encoding.csv')
df_exam = df_raw.copy()

# Step 2. Education 레이블 인코딩
le_edu = LabelEncoder()
df_exam['Education_Label'] = le_edu.fit_transform(df_exam['Education'])

# Step 3. City 원-핫 인코딩
df_exam = pd.get_dummies(df_exam, columns=['City'], dtype=int)

# Step 4. 최종 결과 확인
print('=== [최종 전처리 완료 데이터] ===')
display(df_exam.head())

=== [최종 전처리 완료 데이터] ===


,CustomerID,Education,Satisfaction,PurchaseAmount,Education_Label,City_광주,City_대구,City_부산,City_서울
0,C001,학사,보통,150,3,0,0,0,1
1,C002,석사,만족,230,2,0,0,1,0
2,C003,고졸,불만족,80,0,0,0,0,1
3,C004,박사,매우만족,450,1,0,1,0,0
4,C005,학사,만족,310,3,0,0,1,0


# 📌 범주형 데이터 인코딩 가이드 (get_dummies vs OneHotEncoder vs LabelEncoder)

## 1. 한눈에 보는 인코딩 3종 비교

| 구분 | pd.get_dummies() | OneHotEncoder | LabelEncoder |
| :--- | :--- | :--- | :--- |
| **작동 방식** | 열을 옆으로 펼쳐 0과 1 추가 | 열을 옆으로 펼쳐 0과 1 추가 | 기존 열 안에서 0, 1, 2 숫자로 교체 |
| **변환 결과** | 컬럼 수가 늘어남 | 컬럼 수가 늘어남 | 컬럼 수 유지됨 |
| **주요 대상** | 입력 데이터(X)의 문자열 | 입력 데이터(X)의 문자열 | 정답 데이터(Y) 또는 순서형 변수 |
| **특징 및 용도** | 가장 직관적 / AICE 시험 추천 | 실무 파이프라인 / 배포 환경 | 타겟값(Y) 변환용 |

---

## 2. 상황별 명확한 선택 가이드

1. **AICE 시험 및 빠른 데이터 분석 -> `pd.get_dummies()` 사용**
   - 코드 한 줄로 데이터프레임의 범주형 컬럼을 0과 1의 여러 열로 변환
   - 시험 지문에서 *"범주형 변수를 가변수화(원-핫 인코딩) 하시오"* 시 사용

2. **정답(Target/Y) 데이터 변환 -> `LabelEncoder` 사용**
   - 정답 컬럼(예: 승인/거절, 양성/음성)은 열이 쪼개지면 안 되므로 1개의 열에 0, 1로 변환
   - 예측해야 하는 목표 변수(Y)가 문자열일 때 적용

3. **실무 머신러닝 파이프라인 및 배포 -> `OneHotEncoder` 사용**
   - Train/Test 데이터 간 범주 불일치 방지 및 unseen data 예외 처리 가능
   - Scikit-Learn Pipeline 연동 시 표준 사용법

---

## 3. 🎯 AICE Associate 실기 시험 핵심 공식

* **입력 데이터(X) 문자열 변환:** `pd.get_dummies(df, columns=['컬럼명'])`
* **정답 데이터(Y) 문자열 변환:** `LabelEncoder().fit_transform(df['정답컬럼'])`